# full_3 — Channel-flow (A1) training via `train_channel.py`

Thin wrapper around `train_channel.run_trial`. The heavy lifting (data loading
into RAM, adaptive LR via ReduceLROnPlateau, best-checkpoint S3 save, Trackio
logging) lives in `train_channel.py`. This notebook just sets the config and
runs one trial.

**Set to 3 epochs for a smoke test.** Bump `EPOCHS` below when you're ready
for a full run.

In [1]:
import sys
from pathlib import Path

# make sure we can import train_channel.py from the same directory
_here = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

from train_channel import (
    run_trial,
    EPOCHS,
    BATCH_SIZE,
    NUM_WORKERS,
    S3_MODEL_PREFIX,
    S3_BUCKET,
    STORE_PATH,
)
print("train_channel imported OK")
print(f"  default EPOCHS   = {EPOCHS}")
print(f"  BATCH_SIZE       = {BATCH_SIZE}")
print(f"  NUM_WORKERS      = {NUM_WORKERS}")
print(f"  S3 dataset       = s3://{S3_BUCKET}/{STORE_PATH}")
print(f"  S3 model prefix  = {S3_MODEL_PREFIX}")

train_channel imported OK
  default EPOCHS   = 20
  BATCH_SIZE       = 4
  NUM_WORKERS      = 16
  S3 dataset       = s3://simulations/flow_export/A1
  S3 model prefix  = flow_models/channel


## Config

In [2]:
# --- run config -------------------------------------------------------
EPOCHS_RUN = 3                       # smoke test; bump for a full run
LR_INIT    = 1e-3                    # initial LR (adaptive via ReduceLROnPlateau)
PROJECT    = "channel_full3"         # Trackio project name
LABEL      = "full3-smoke"           # trial label for Trackio

# --- architecture knobs (sweepable) -----------------------------------
# DataNet from full_2.ipynb: 1 pressure channel -> 3 velocity channels.
# c         : channel width of the 3D conv core
# n_blocks  : number of ConvBlock3D(2xConv3d+SiLU) blocks in the core
# dy        : depth dimension (129 for the A1 channel grid)
ARCH = {
    "c": 64,
    "n_blocks": 3,
    "dy": 129,
}

print(f"config: epochs={EPOCHS_RUN}  lr={LR_INIT}  arch={ARCH}")

config: epochs=3  lr=0.001  arch={'c': 64, 'n_blocks': 3, 'dy': 129}


## Run the trial

This loads the A1 dataset into RAM (assumed to fit), builds the `DataNet`,
trains for `EPOCHS_RUN` epochs with an adaptive LR, and keeps only the best
validation checkpoint on S3.

In [ ]:
result = run_trial(
    ARCH,
    epochs=EPOCHS_RUN,
    lr=LR_INIT,
    trackio_project=PROJECT,
    trial_label=LABEL,
    log=True,
)

[trial] {'c': 64, 'n_blocks': 3, 'dy': 129}  lr=0.001  epochs=3
* Trackio project initialized: channel_full3
* Trackio metrics logged to: /home/chancrin/.cache/huggingface/trackio
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: dainty-sunset-0


  T=4001  train=3136  val=392
  model: 0.67M params


## Result

In [ ]:
import json

# strip the live model out of the printable summary
summary = {k: v for k, v in result.items() if k != "model"}
print(json.dumps(summary, indent=2))
print()
print(f"best val MSE : {result['val_mse']:.6f}")
print(f"best epoch   : {result['best_epoch']}")
print(f"n params     : {result['n_params']:,}")
print(f"checkpoint   : s3://simulations/{result['s3_prefix']}/ckpt_best.pt")

In [ ]:
# optional: save a local copy of the best model for quick inspection
import torch
torch.save(
    {"state_dict": result["model"].state_dict(),
     "config": {k: v for k, v in result.items() if k != "model"}},
    "full3_best.pth",
)
print("saved full3_best.pth")